In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
from pathlib import Path
import glob
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils import data
from torch.utils.data import Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
# M2 TCGA-only 데이터 로드
# 입력: WSI tile image paths + tile 좌표 + slide 전체 크기 + basic clinical(age, sex) + multi-horizon survival label/mask
# Clinical 입력에서는 pathologic_stage/T/N/M/tumor_grade를 사용하지 않습니다.

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"
CLINICAL_PATH = DST_PATH / "Clinical"

M2_OUTPUT_PATH = Path("outputs/M2")
M2_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "TCGA_PAAD"
PATCH_INPUT_SIZE = 512
PATCH_MEAN = (0.485, 0.456, 0.406)
PATCH_STD = (0.229, 0.224, 0.225)
TEST_SIZE = 0.2
VALID_SIZE = 0.2

MONTH_DAYS = 30.4375
HORIZON_MONTHS = [6, 12, 18, 24]
HORIZON_DAYS = np.array([m * MONTH_DAYS for m in HORIZON_MONTHS], dtype=np.float32)
HORIZON_NAMES = [f"dead_by_{m}m" for m in HORIZON_MONTHS]
CLINICAL_FEATURE_COLUMNS = ["age_years_z", "sex_male", "sex_female"]


def load_clinical_json(dataset: str, case_id: str) -> dict:
    clinical_json_path = CLINICAL_PATH / dataset / f"{case_id}_clinical.json"
    if not clinical_json_path.exists():
        return {}
    with open(clinical_json_path, "r", encoding="utf-8") as f:
        clinical_json = json.load(f)
    return clinical_json.get("clinical", {})


def make_horizon_label_mask(os_time_days: float, os_event: int) -> tuple[np.ndarray, np.ndarray]:
    y = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    mask = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    if pd.isna(os_time_days) or pd.isna(os_event):
        return y, mask

    os_time_days = float(os_time_days)
    os_event = int(os_event)
    for i, horizon in enumerate(HORIZON_DAYS):
        if os_event == 1 and os_time_days <= float(horizon):
            y[i] = 1.0
            mask[i] = 1.0
        elif os_time_days >= float(horizon):
            y[i] = 0.0
            mask[i] = 1.0
        else:
            y[i] = 0.0
            mask[i] = 0.0
    return y, mask


def get_patch_padding(image_size: int = PATCH_INPUT_SIZE) -> tuple[int, int, int, int]:
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    target_size = int(np.ceil(image_size / patch_size) * patch_size)
    pad_total = max(0, target_size - image_size)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return (pad_left, pad_left, pad_right, pad_right)


def get_model_input_size(image_size: int = PATCH_INPUT_SIZE) -> int:
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    return int(np.ceil(image_size / patch_size) * patch_size)


def get_train_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def load_tcga_case_tiles(summary_row: pd.Series) -> tuple[pd.DataFrame, dict]:
    case_id = str(summary_row["case_id"])
    case_dir = Path(summary_row["case_dir"])
    metadata_path = Path(summary_row["metadata_path"])
    tile_df = pd.read_csv(metadata_path)

    tile_df["x"] = tile_df["x_level0"].astype(int)
    tile_df["y"] = tile_df["y_level0"].astype(int)
    slide_width = int(summary_row["slide_width"])
    slide_height = int(summary_row["slide_height"])

    tile_df["tile_path"] = tile_df["tile_path"].astype(str)
    tile_df = tile_df[tile_df["tile_path"].map(lambda x: Path(x).exists())].copy()
    tile_df["slide_width"] = slide_width
    tile_df["slide_height"] = slide_height

    source_size = tile_df["source_tile_size_px"].astype(float)
    tile_df["x_norm"] = tile_df["x"].astype(float) / slide_width
    tile_df["y_norm"] = tile_df["y"].astype(float) / slide_height
    tile_df["x_center_norm"] = (tile_df["x"].astype(float) + source_size / 2) / slide_width
    tile_df["y_center_norm"] = (tile_df["y"].astype(float) + source_size / 2) / slide_height
    tile_df["w_norm"] = source_size / slide_width
    tile_df["h_norm"] = source_size / slide_height

    clinical = load_clinical_json(DATASET_NAME, case_id)
    os_time = clinical.get("os_time_days", tile_df["OS_time"].iloc[0] if "OS_time" in tile_df.columns else np.nan)
    os_event = clinical.get("os_event", tile_df["OS_event"].iloc[0] if "OS_event" in tile_df.columns else np.nan)
    age_years = clinical.get("age_years", np.nan)
    sex = str(clinical.get("sex", "unknown")).lower()

    y, mask = make_horizon_label_mask(os_time, os_event)
    case_record = {
        "dataset": DATASET_NAME,
        "case_id": case_id,
        "case_dir": case_dir.as_posix(),
        "metadata_path": metadata_path.as_posix(),
        "n_tiles": int(len(tile_df)),
        "slide_width": slide_width,
        "slide_height": slide_height,
        "os_time_days": float(os_time) if pd.notna(os_time) else np.nan,
        "os_event": int(os_event) if pd.notna(os_event) else np.nan,
        "age_years": float(age_years) if pd.notna(age_years) else np.nan,
        "sex": sex,
        "known_horizon_count": int(mask.sum()),
    }
    for i, name in enumerate(HORIZON_NAMES):
        case_record[name] = float(y[i])
        case_record[f"mask_{name}"] = float(mask[i])
    return tile_df, case_record


summary_path = IMAGE_PATH / DATASET_NAME / "tile_summary.csv"
summary_df = pd.read_csv(summary_path)
summary_df = summary_df[summary_df["metadata_path"].notna()].copy()

case_records = []
tile_index_list = []

for _, row in tqdm(summary_df.iterrows(), total=len(summary_df), desc=f"Load {DATASET_NAME} metadata"):
    metadata_path = Path(row["metadata_path"])
    if not metadata_path.exists():
        continue
    tile_df, case_record = load_tcga_case_tiles(row)
    if case_record["n_tiles"] > 0:
        tile_df["dataset"] = DATASET_NAME
        tile_df["case_id"] = case_record["case_id"]
        tile_index_list.append(tile_df)
    case_records.append(case_record)

all_slide_df = pd.DataFrame(case_records)
tile_index_df = pd.concat(tile_index_list, ignore_index=True)

eligible_mask = (
    all_slide_df["os_time_days"].notna()
    & all_slide_df["os_event"].notna()
    & all_slide_df["known_horizon_count"].gt(0)
    & all_slide_df["n_tiles"].gt(0)
    & all_slide_df["age_years"].notna()
    & all_slide_df["sex"].isin(["male", "female"])
)
excluded_df = all_slide_df[~eligible_mask].copy()
if not excluded_df.empty:
    excluded_df["exclude_reason"] = np.select(
        [
            excluded_df["n_tiles"].le(0),
            excluded_df["os_time_days"].isna(),
            excluded_df["os_event"].isna(),
            excluded_df["known_horizon_count"].le(0),
            excluded_df["age_years"].isna(),
            ~excluded_df["sex"].isin(["male", "female"]),
        ],
        ["no_tiles", "missing_os_time", "missing_os_event", "no_known_horizon", "missing_age", "missing_sex"],
        default="unknown",
    )

slide_df = all_slide_df[eligible_mask].copy()
slide_df["os_event"] = slide_df["os_event"].astype(int)
tile_index_df = tile_index_df[tile_index_df["case_id"].isin(slide_df["case_id"])].copy()

train_valid_df, test_df = train_test_split(slide_df, test_size=TEST_SIZE, random_state=SEED, stratify=slide_df["os_event"])
train_df, valid_df = train_test_split(train_valid_df, test_size=VALID_SIZE, random_state=SEED, stratify=train_valid_df["os_event"])

age_mean = float(train_df["age_years"].mean())
age_std = float(train_df["age_years"].std(ddof=0))
age_std = age_std if age_std > 0 else 1.0
clinical_stats = {"age_mean": age_mean, "age_std": age_std, "sex_encoding": ["sex_male", "sex_female"]}


def add_clinical_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["age_years_z"] = (df["age_years"].astype(float) - age_mean) / age_std
    df["sex_male"] = df["sex"].eq("male").astype(float)
    df["sex_female"] = df["sex"].eq("female").astype(float)
    return df

train_df = add_clinical_features(train_df)
valid_df = add_clinical_features(valid_df)
test_df = add_clinical_features(test_df)
slide_df = pd.concat([train_df, valid_df, test_df], axis=0).sort_index()

slide_df.to_csv(M2_OUTPUT_PATH / "m2_tcga_horizon_clinical_slide_manifest.csv", index=False)
tile_index_df.to_csv(M2_OUTPUT_PATH / "m2_tcga_horizon_tile_index.csv", index=False)
excluded_df.to_csv(M2_OUTPUT_PATH / "m2_tcga_horizon_excluded_cases.csv", index=False)
with open(M2_OUTPUT_PATH / "m2_clinical_stats.json", "w", encoding="utf-8") as f:
    json.dump(clinical_stats, f, indent=2, ensure_ascii=False)

print("all_slide_df:", all_slide_df.shape)
print("slide_df:", slide_df.shape)
print("excluded_df:", excluded_df.shape)
print("tile_index_df:", tile_index_df.shape)
print("clinical stats:", clinical_stats)

display(slide_df[["case_id", "age_years", "sex"] + CLINICAL_FEATURE_COLUMNS].head())

horizon_summary = []
for name in HORIZON_NAMES:
    mask_col = f"mask_{name}"
    known = slide_df[mask_col].eq(1)
    horizon_summary.append({
        "horizon": name,
        "known_n": int(known.sum()),
        "dead_n": int(slide_df.loc[known, name].sum()),
        "alive_n": int(known.sum() - slide_df.loc[known, name].sum()),
        "unknown_n": int((~known).sum()),
        "dead_rate_known": float(slide_df.loc[known, name].mean()) if known.any() else np.nan,
    })
display(pd.DataFrame(horizon_summary))


class M2SlideDataset(Dataset):
    """TCGA pathology + basic clinical dataset for multi-horizon MIL training."""

    def __init__(self, slide_manifest: pd.DataFrame, tile_index: pd.DataFrame):
        self.slide_manifest = slide_manifest.reset_index(drop=True).copy()
        self.tile_groups = {
            case_id: group.sort_values(["y", "x"]).reset_index(drop=True)
            for case_id, group in tile_index.groupby("case_id")
        }

    def __len__(self):
        return len(self.slide_manifest)

    def __getitem__(self, idx):
        row = self.slide_manifest.iloc[idx]
        tiles = self.tile_groups[row["case_id"]]
        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        label = row[HORIZON_NAMES].to_numpy(np.float32)
        mask = row[[f"mask_{name}" for name in HORIZON_NAMES]].to_numpy(np.float32)
        clinical_features = row[CLINICAL_FEATURE_COLUMNS].to_numpy(np.float32)
        slide_size = np.array([row["slide_width"], row["slide_height"]], dtype=np.float32)
        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "slide_size": torch.from_numpy(slide_size),
            "clinical_features": torch.from_numpy(clinical_features),
            "horizon_months": torch.tensor(HORIZON_MONTHS, dtype=torch.float32),
            "label": torch.from_numpy(label),
            "mask": torch.from_numpy(mask),
            "os_time_days": torch.tensor(float(row["os_time_days"]), dtype=torch.float32),
            "os_event": torch.tensor(int(row["os_event"]), dtype=torch.long),
        }


class M2TileDataset(Dataset):
    def __init__(self, tile_index: pd.DataFrame, transform=None):
        self.tile_index = tile_index.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.tile_index)

    def __getitem__(self, idx):
        row = self.tile_index.iloc[idx]
        image = Image.open(row["tile_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        coords = row[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        return {"image": image, "coords": torch.from_numpy(coords), "case_id": row["case_id"], "tile_path": row["tile_path"]}


train_case_ids = set(train_df["case_id"])
valid_case_ids = set(valid_df["case_id"])
test_case_ids = set(test_df["case_id"])
assert len(train_case_ids & valid_case_ids) == 0
assert len(train_case_ids & test_case_ids) == 0
assert len(valid_case_ids & test_case_ids) == 0

train_dataset = M2SlideDataset(train_df, tile_index_df)
valid_dataset = M2SlideDataset(valid_df, tile_index_df)
test_dataset = M2SlideDataset(test_df, tile_index_df)

train_tile_dataset = M2TileDataset(tile_index_df[tile_index_df["case_id"].isin(train_case_ids)], transform=get_train_patch_transform())
valid_tile_dataset = M2TileDataset(tile_index_df[tile_index_df["case_id"].isin(valid_case_ids)], transform=get_eval_patch_transform())
test_tile_dataset = M2TileDataset(tile_index_df[tile_index_df["case_id"].isin(test_case_ids)], transform=get_eval_patch_transform())

split_df = slide_df[["dataset", "case_id", "os_time_days", "os_event", "age_years", "sex"] + CLINICAL_FEATURE_COLUMNS + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].copy()
split_df["split"] = "unused"
split_df.loc[split_df["case_id"].isin(train_case_ids), "split"] = "train"
split_df.loc[split_df["case_id"].isin(valid_case_ids), "split"] = "valid"
split_df.loc[split_df["case_id"].isin(test_case_ids), "split"] = "test"
split_df.to_csv(M2_OUTPUT_PATH / "m2_tcga_horizon_clinical_case_splits.csv", index=False)

print("slide splits:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("tile splits:", len(train_tile_dataset), len(valid_tile_dataset), len(test_tile_dataset))
display(pd.crosstab(split_df["split"], split_df["os_event"]))

sample = train_dataset[0]
print("sample case:", sample["case_id"])
print("n_tiles:", len(sample["tile_paths"]))
print("clinical_features:", sample["clinical_features"])
print("label:", sample["label"])
print("mask:", sample["mask"])
print("model input size:", get_model_input_size(PATCH_INPUT_SIZE), "padding:", get_patch_padding(PATCH_INPUT_SIZE))


In [ ]:
# M2 모델 정의 및 UNI/UNI2-h 로드, loss/optimizer 구성
# 구조: frozen UNI/UNI2-h feature extractor + coordinate spatial embedding + attention MIL + clinical embedding

import timm
from huggingface_hub import hf_hub_download

from scripts.models.m2_pathology_clinical_mil import (
    M2ModelConfig,
    PathologyClinicalMIL,
    masked_bce_with_logits,
    sample_tiles,
    build_optimizer,
)

print("device:", device)

MAX_TILES_PER_SLIDE = 10000
FEATURE_BATCH_SIZE = 64
SPATIAL_DIM = 128
CLINICAL_EMBED_DIM = 64
FUSION_DIM = 512
MIL_HIDDEN_DIM = 256
DROPOUT = 0.25
N_OUTPUTS = len(HORIZON_NAMES)
CLINICAL_DIM = len(CLINICAL_FEATURE_COLUMNS)
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

FEATURE_EXTRACTOR_NAME = "UNI2-h"
UNI_BACKBONES = {
    "UNI": {
        "repo_id": "MahmoodLab/UNI",
        "filename": "pytorch_model.bin",
        "feature_dim": 1024,
        "timm_kwargs": {
            "model_name": "vit_large_patch16_224",
            "img_size": 224,
            "patch_size": 16,
            "init_values": 1e-5,
            "num_classes": 0,
            "dynamic_img_size": True,
        },
    },
    "UNI2-h": {
        "repo_id": "MahmoodLab/UNI2-h",
        "filename": "pytorch_model.bin",
        "feature_dim": 1536,
        "timm_kwargs": {
            "model_name": "vit_giant_patch14_224",
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        },
    },
}


def load_uni_feature_extractor(name: str = FEATURE_EXTRACTOR_NAME, device: torch.device = device) -> tuple[nn.Module, int]:
    cfg = UNI_BACKBONES[name]
    print(f"Loading {name}: {cfg['repo_id']} / {cfg['timm_kwargs']['model_name']}")
    model = timm.create_model(pretrained=False, **cfg["timm_kwargs"])
    weight_path = hf_hub_download(repo_id=cfg["repo_id"], filename=cfg["filename"], local_files_only=False)
    state_dict = torch.load(weight_path, map_location="cpu")
    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    print("missing:", len(missing), "unexpected:", len(unexpected))
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(device)
    feature_dim = int(getattr(model, "num_features", cfg["feature_dim"]))
    print(f"{name} loaded. feature_dim={feature_dim}")
    return model, feature_dim


def compute_horizon_pos_weight(train_manifest: pd.DataFrame, horizon_names: list[str], device: torch.device) -> torch.Tensor:
    weights = []
    rows = []
    for name in horizon_names:
        mask_col = f"mask_{name}"
        known = train_manifest[mask_col].eq(1)
        pos = float(train_manifest.loc[known, name].sum())
        neg = float(known.sum() - pos)
        weight = neg / max(pos, 1.0)
        weights.append(weight)
        rows.append({"horizon": name, "known": int(known.sum()), "positive_dead": int(pos), "negative_alive": int(neg), "pos_weight": weight})
    pos_weight_df = pd.DataFrame(rows)
    display(pos_weight_df)
    return torch.tensor(weights, dtype=torch.float32, device=device)


pos_weight = compute_horizon_pos_weight(train_df, HORIZON_NAMES, device)

feature_extractor, M2_FEATURE_DIM = load_uni_feature_extractor(name=FEATURE_EXTRACTOR_NAME, device=device)
FEATURE_EXTRACTOR_PATCH_SIZE = int(UNI_BACKBONES[FEATURE_EXTRACTOR_NAME]["timm_kwargs"]["patch_size"])
print("FEATURE_EXTRACTOR_PATCH_SIZE:", FEATURE_EXTRACTOR_PATCH_SIZE)
print("PATCH_INPUT_SIZE:", PATCH_INPUT_SIZE, "-> model input size:", get_model_input_size(PATCH_INPUT_SIZE))
print("PATCH_PADDING:", get_patch_padding(PATCH_INPUT_SIZE))

train_tile_dataset.transform = get_train_patch_transform()
valid_tile_dataset.transform = get_eval_patch_transform()
test_tile_dataset.transform = get_eval_patch_transform()

m2_config = M2ModelConfig(
    feature_dim=M2_FEATURE_DIM,
    coord_dim=6,
    clinical_dim=CLINICAL_DIM,
    spatial_dim=SPATIAL_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    fusion_dim=FUSION_DIM,
    mil_hidden_dim=MIL_HIDDEN_DIM,
    n_outputs=N_OUTPUTS,
    dropout=DROPOUT,
    max_tiles=MAX_TILES_PER_SLIDE,
    feature_batch_size=FEATURE_BATCH_SIZE,
    freeze_feature_extractor=True,
)


def m2_loss_fn(logits: torch.Tensor, labels: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
    return masked_bce_with_logits(logits=logits, labels=labels, masks=masks, pos_weight=pos_weight)


def initialize_m2_model(feature_extractor: nn.Module, feature_dim: int = M2_FEATURE_DIM) -> tuple[PathologyClinicalMIL, torch.optim.Optimizer]:
    config = M2ModelConfig(
        feature_dim=feature_dim,
        coord_dim=6,
        clinical_dim=CLINICAL_DIM,
        spatial_dim=SPATIAL_DIM,
        clinical_embed_dim=CLINICAL_EMBED_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
    )
    model = PathologyClinicalMIL(feature_extractor=feature_extractor, config=config).to(device)
    optimizer = build_optimizer(model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    return model, optimizer


model, optimizer = initialize_m2_model(feature_extractor, feature_dim=M2_FEATURE_DIM)
print("M2 model initialized.")
print("FEATURE_EXTRACTOR_NAME:", FEATURE_EXTRACTOR_NAME)
print("M2_FEATURE_DIM:", M2_FEATURE_DIM)
print("CLINICAL_DIM:", CLINICAL_DIM, CLINICAL_FEATURE_COLUMNS)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("MAX_TILES_PER_SLIDE:", MAX_TILES_PER_SLIDE)
print("FEATURE_BATCH_SIZE:", FEATURE_BATCH_SIZE)
print("N_OUTPUTS:", N_OUTPUTS, HORIZON_NAMES)


In [ ]:
# M2 학습 하이퍼파라미터 정의

MODEL_PATH = Path("../../model")
M2_MODEL_DIR = MODEL_PATH / "pancreatic_cancer_pathology" / "M2" / FEATURE_EXTRACTOR_NAME
M2_MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 50
PATIENCE = 15
MIN_DELTA = 1e-4
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16
SLIDE_BATCH_SIZE = 1
LOG_EVERY_EPOCHS = 10
SAVE_EVERY_EPOCHS = 10

BEST_CHECKPOINT_PATH = M2_MODEL_DIR / "best_checkpoint.pt"
LAST_CHECKPOINT_PATH = M2_MODEL_DIR / "last_checkpoint.pt"
TRAIN_LOG_PATH = M2_MODEL_DIR / "training_log.csv"
CONFIG_PATH = M2_MODEL_DIR / "training_config.json"

training_config = {
    "model": "M2_pathology_basic_clinical",
    "dataset": DATASET_NAME,
    "feature_extractor": FEATURE_EXTRACTOR_NAME,
    "clinical_features": CLINICAL_FEATURE_COLUMNS,
    "clinical_stats": clinical_stats,
    "patch_input_size": PATCH_INPUT_SIZE,
    "model_input_size": get_model_input_size(PATCH_INPUT_SIZE),
    "patch_padding": get_patch_padding(PATCH_INPUT_SIZE),
    "feature_extractor_patch_size": FEATURE_EXTRACTOR_PATCH_SIZE,
    "effective_mpp": 1.0 * 1024 / PATCH_INPUT_SIZE,
    "max_tiles_per_slide": MAX_TILES_PER_SLIDE,
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "horizon_months": HORIZON_MONTHS,
    "horizon_names": HORIZON_NAMES,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "spatial_dim": SPATIAL_DIM,
    "clinical_embed_dim": CLINICAL_EMBED_DIM,
    "fusion_dim": FUSION_DIM,
    "mil_hidden_dim": MIL_HIDDEN_DIM,
    "use_amp": USE_AMP,
    "amp_dtype": str(AMP_DTYPE),
    "seed": SEED,
    "model_dir": M2_MODEL_DIR.as_posix(),
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

print("M2_MODEL_DIR:", M2_MODEL_DIR)
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("EPOCHS:", EPOCHS)
print("PATIENCE:", PATIENCE)
print("USE_AMP:", USE_AMP)
print("scheduler:", scheduler.__class__.__name__)


In [ ]:
# M2 학습 및 validation
# tqdm progress, validation tqdm, checkpoint, scheduler, 10 epoch마다 학습 추이 확인

from PIL import Image
import matplotlib.pyplot as plt


def load_tile_tensor_batch(tile_paths: list[str], transform, device: torch.device) -> torch.Tensor:
    images = []
    for path in tile_paths:
        image = Image.open(path).convert("RGB")
        images.append(transform(image))
    return torch.stack(images, dim=0).to(device, non_blocking=True)


def prepare_slide_batch(sample: dict, training: bool) -> dict:
    transform = get_train_patch_transform() if training else get_eval_patch_transform()
    selected_paths, selected_coords, selected_indices = sample_tiles(
        sample["tile_paths"], sample["coords"], max_tiles=MAX_TILES_PER_SLIDE, training=training
    )
    tile_images = load_tile_tensor_batch(selected_paths, transform=transform, device=device)
    return {
        "tile_images": tile_images,
        "coords": selected_coords.to(device, non_blocking=True),
        "clinical_features": sample["clinical_features"].to(device, non_blocking=True).float(),
        "labels": sample["label"].unsqueeze(0).to(device, non_blocking=True).float(),
        "masks": sample["mask"].unsqueeze(0).to(device, non_blocking=True).float(),
        "case_id": sample["case_id"],
        "n_tiles": len(selected_paths),
    }


def masked_binary_accuracy(logits: torch.Tensor, labels: torch.Tensor, masks: torch.Tensor) -> float:
    with torch.no_grad():
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct = ((preds == labels).float() * masks).sum()
        denom = masks.sum().clamp_min(1.0)
        return float((correct / denom).detach().cpu())


def run_one_epoch(dataset, training: bool, epoch: int) -> dict:
    if training:
        model.train()
        model.feature_extractor.eval()
    else:
        model.eval()

    total_loss = 0.0
    total_acc_weighted = 0.0
    total_mask = 0.0
    horizon_loss_sum = torch.zeros(N_OUTPUTS, dtype=torch.float64)
    horizon_mask_sum = torch.zeros(N_OUTPUTS, dtype=torch.float64)

    indices = list(range(len(dataset)))
    if training:
        random.shuffle(indices)

    desc = f"Epoch {epoch:03d} train" if training else f"Epoch {epoch:03d} valid"
    pbar = tqdm(indices, desc=desc, leave=False)

    for idx in pbar:
        sample = dataset[idx]
        batch = prepare_slide_batch(sample, training=training)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    outputs = model(batch["tile_images"], batch["coords"], batch["clinical_features"])
                    loss = m2_loss_fn(outputs["logits"], batch["labels"], batch["masks"])
            else:
                outputs = model(batch["tile_images"], batch["coords"], batch["clinical_features"])
                loss = m2_loss_fn(outputs["logits"], batch["labels"], batch["masks"])

            if training:
                loss.backward()
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM)
                optimizer.step()

        with torch.no_grad():
            mask_count = float(batch["masks"].sum().detach().cpu())
            acc = masked_binary_accuracy(outputs["logits"], batch["labels"], batch["masks"])
            total_loss += float(loss.detach().cpu()) * max(mask_count, 1.0)
            total_acc_weighted += acc * max(mask_count, 1.0)
            total_mask += max(mask_count, 1.0)

            raw_loss = torch.nn.functional.binary_cross_entropy_with_logits(
                outputs["logits"].detach().float(),
                batch["labels"].detach().float(),
                pos_weight=pos_weight.detach().float(),
                reduction="none",
            )
            horizon_loss_sum += ((raw_loss * batch["masks"]).sum(dim=0).detach().cpu().double())
            horizon_mask_sum += batch["masks"].sum(dim=0).detach().cpu().double()

        running_loss = total_loss / max(total_mask, 1.0)
        running_acc = total_acc_weighted / max(total_mask, 1.0)
        pbar.set_postfix({
            "avg_loss": f"{running_loss:.4f}",
            "avg_acc": f"{running_acc:.3f}",
            "last_loss": f"{float(loss.detach().cpu()):.4f}",
            "tiles": batch["n_tiles"],
        })

        del batch, outputs, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    avg_loss = total_loss / max(total_mask, 1.0)
    avg_acc = total_acc_weighted / max(total_mask, 1.0)
    horizon_loss = (horizon_loss_sum / horizon_mask_sum.clamp_min(1.0)).numpy()
    metrics = {"loss": avg_loss, "masked_acc": avg_acc}
    for name, value in zip(HORIZON_NAMES, horizon_loss):
        metrics[f"loss_{name}"] = float(value)
    return metrics


def get_trainable_state_dict(model: torch.nn.Module) -> dict:
    return {key: value.detach().cpu() for key, value in model.state_dict().items() if not key.startswith("feature_extractor.")}


def save_checkpoint(path: Path, epoch: int, best_valid_loss: float, history: list[dict]):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": get_trainable_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_valid_loss": best_valid_loss,
        "history": history,
        "config": training_config,
        "horizon_names": HORIZON_NAMES,
        "clinical_feature_columns": CLINICAL_FEATURE_COLUMNS,
        "clinical_stats": clinical_stats,
        "note": "feature_extractor weights are excluded; reload UNI/UNI2-h before loading this checkpoint with strict=False.",
    }
    torch.save(checkpoint, path)


def plot_training_history(history: list[dict]):
    hist_df = pd.DataFrame(history)
    display(hist_df.tail(10))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
    axes[0].plot(hist_df["epoch"], hist_df["valid_loss"], label="valid")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(hist_df["epoch"], hist_df["train_acc"], label="train")
    axes[1].plot(hist_df["epoch"], hist_df["valid_acc"], label="valid")
    axes[1].set_title("Masked Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[2].plot(hist_df["epoch"], hist_df["lr"], label="lr")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("epoch")
    axes[2].set_yscale("log")
    axes[2].legend()
    plt.tight_layout()
    plt.show()


history = []
best_valid_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch(train_dataset, training=True, epoch=epoch)
    valid_metrics = run_one_epoch(valid_dataset, training=False, epoch=epoch)
    scheduler.step(valid_metrics["loss"])
    current_lr = optimizer.param_groups[0]["lr"]

    log_row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "valid_loss": valid_metrics["loss"],
        "train_acc": train_metrics["masked_acc"],
        "valid_acc": valid_metrics["masked_acc"],
        "lr": current_lr,
    }
    for name in HORIZON_NAMES:
        log_row[f"train_loss_{name}"] = train_metrics[f"loss_{name}"]
        log_row[f"valid_loss_{name}"] = valid_metrics[f"loss_{name}"]
    history.append(log_row)
    pd.DataFrame(history).to_csv(TRAIN_LOG_PATH, index=False)

    improved = valid_metrics["loss"] < (best_valid_loss - MIN_DELTA)
    if improved:
        best_valid_loss = valid_metrics["loss"]
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(BEST_CHECKPOINT_PATH, epoch, best_valid_loss, history)
        checkpoint_msg = "best saved"
    else:
        epochs_without_improvement += 1
        checkpoint_msg = f"no improve {epochs_without_improvement}/{PATIENCE}"

    save_checkpoint(LAST_CHECKPOINT_PATH, epoch, best_valid_loss, history)
    if epoch % SAVE_EVERY_EPOCHS == 0:
        save_checkpoint(M2_MODEL_DIR / f"checkpoint_epoch_{epoch:03d}.pt", epoch, best_valid_loss, history)

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_metrics['loss']:.4f} valid_loss={valid_metrics['loss']:.4f} | "
        f"train_acc={train_metrics['masked_acc']:.3f} valid_acc={valid_metrics['masked_acc']:.3f} | "
        f"lr={current_lr:.2e} | best_epoch={best_epoch} ({checkpoint_msg})"
    )

    if epoch % LOG_EVERY_EPOCHS == 0 or epoch == 1:
        plot_training_history(history)

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: {PATIENCE} epochs without validation loss improvement.")
        break

print("Training finished.")
print("best_valid_loss:", best_valid_loss)
print("best_epoch:", best_epoch)
print("best_checkpoint:", BEST_CHECKPOINT_PATH)
print("last_checkpoint:", LAST_CHECKPOINT_PATH)
